In [21]:
import sys
import numpy as np # type: ignore
import os
from torch.utils.data import DataLoader # type: ignore

# Aggiungi il percorso del progetto al sys.path
project_path = 'c:\\Users\\luca-\\OneDrive\\Desktop\\University\\#MAGISTRALE\\2 ANNO\\1 SEMESTRE\\DEEP LEARNING\\project\\informer_project_dl\informer2020'
if project_path not in sys.path:
    sys.path.append(project_path)

import torch # type: ignore
import matplotlib.pyplot as plt # type: ignore
from utils.tools import dotdict
from exp.exp_informer import Exp_Informer

print(torch.__file__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

args = dotdict()

args.model = 'informer' # model of experiment, options: [informer, informerstack, informerlight(TBD)]

c:\Users\luca-\OneDrive\Desktop\University\#MAGISTRALE\2 ANNO\1 SEMESTRE\DEEP LEARNING\project\informer_project_dl\.venv\lib\site-packages\torch\__init__.py
Torch version: 1.8.0+cu111
CUDA available: True


Dataset

In [22]:
args.data = 'custom' # data
args.root_path = '../data/final/' # root path of data file
args.data_path = 'B101_50khz_downsampled.csv' # data file
args.features = 'M' # forecasting task, options:[M, S, MS]; M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate
args.target = 'Channel62'
args.freq = 't' # freq for time features encoding, options:[s:secondly, t:minutely, h:hourly, d:daily, b:business days, w:weekly, m:monthly], you can also use more detailed freq like 15min or 3h
args.checkpoints = '../informer_checkpoints' # location of model checkpoints

In [23]:
# === Model ===
args.seq_len = 24 # input sequence length of Informer encoder
args.label_len = 24 # start token length of Informer decoder
args.pred_len = 24 # prediction sequence length

args.enc_in = 62 # encoder input size
args.dec_in = 62 # decoder input size
args.c_out = 62 # output size
args.factor = 5 # probsparse attn factor
args.d_model = 512 # dimension of model
args.n_heads = 8 # num of heads
args.e_layers = 3 # num of encoder layers
args.d_layers = 2 # num of decoder layers
args.d_ff = 2048 # dimension of fcn in model
args.dropout = 0.05 # dropout
args.attn = 'prob' # attention used in encoder, options:[prob, full]
args.embed = 'timeF' # time features encoding, options:[timeF, fixed, learned]
args.activation = 'gelu' # activation
args.distil = True # whether to use distilling in encoder
args.output_attention = False # whether to output attention in ecoder
args.mix = True
args.padding = 0

In [24]:
# === Optimizer ===
args.batch_size = 64
args.learning_rate = 0.0001
args.loss = 'mse'
args.lradj = 'type2'

args.use_amp = False # whether to use automatic mixed precision training

args.num_workers = 0
args.itr = 1
args.train_epochs = 10
args.patience = 3
args.des = 'exp'

args.use_gpu = True if torch.cuda.is_available() else False
args.gpu = 0

args.use_multi_gpu = False
args.devices = '0,1,2,3'

args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(' ','')
    device_ids = args.devices.split(',')
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

In [25]:
#Set augments by using data name
data_parser = {
    'ETTh1':{'data':'ETTh1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTh2':{'data':'ETTh2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm1':{'data':'ETTm1.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'ETTm2':{'data':'ETTm2.csv','T':'OT','M':[7,7,7],'S':[1,1,1],'MS':[7,7,1]},
    'custom': {'data': 'B101_50khz_downsampled.csv', 'T': 'Channel62', 'M': [62, 62, 62], 'S': [1, 1, 1], 'MS': [62, 62, 1]},

}
if args.data in data_parser.keys():
    data_info = data_parser[args.data]
    args.data_path = data_info['data']
    args.target = data_info['T']
    args.enc_in, args.dec_in, args.c_out = data_info[args.features]
    
args.detail_freq = args.freq
args.freq = args.freq[-1:]

print('Args in experiment:')
print(args)

Exp = Exp_Informer

Args in experiment:
{'model': 'informer', 'data': 'custom', 'root_path': '../data/final/', 'data_path': 'B101_50khz_downsampled.csv', 'features': 'M', 'target': 'Channel62', 'freq': 't', 'checkpoints': '../informer_checkpoints', 'seq_len': 24, 'label_len': 24, 'pred_len': 24, 'enc_in': 62, 'dec_in': 62, 'c_out': 62, 'factor': 5, 'd_model': 512, 'n_heads': 8, 'e_layers': 3, 'd_layers': 2, 'd_ff': 2048, 'dropout': 0.05, 'attn': 'prob', 'embed': 'timeF', 'activation': 'gelu', 'distil': True, 'output_attention': False, 'mix': True, 'padding': 0, 'batch_size': 64, 'learning_rate': 0.0001, 'loss': 'mse', 'lradj': 'type2', 'use_amp': False, 'num_workers': 0, 'itr': 1, 'train_epochs': 10, 'patience': 3, 'des': 'exp', 'use_gpu': True, 'gpu': 0, 'use_multi_gpu': False, 'devices': '0,1,2,3', 'detail_freq': 't'}


Train

In [ ]:
for ii in range(args.itr):
    # setting record of experiments
    setting = '{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_at{}_fc{}_eb{}_dt{}_mx{}_{}_{}'.format(args.model, args.data, args.features,
                args.seq_len, args.label_len, args.pred_len,
                args.d_model, args.n_heads, args.e_layers, args.d_layers, args.d_ff, args.attn, args.factor, args.embed, args.distil, args.mix, args.des, ii)

    # set experiments
    exp = Exp(args)

    # train
    print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
    exp.train(setting)

    # test
    print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
    exp.test(setting)

    torch.cuda.empty_cache()

Use GPU: cuda:0
>>>>>>>start training : informer_custom_ftM_sl24_ll24_pl24_dm512_nh8_el3_dl2_df2048_atprob_fc5_ebtimeF_dtTrue_mxTrue_exp_0>>>>>>>>>>>>>>>>>>>>>>>>>>


Predizioni

In [ ]:
# set saved model path
setting = 'informer_custom_ftM_sl24_ll24_pl24_dm512_nh8_el3_dl2_df2048_atprob_fc5_ebtimeF_dtTrue_mxTrue_exp_0'
# path = os.path.join(args.checkpoints,setting,'checkpoint.pth')

# If you already have a trained model, you can set the arguments and model path, then initialize a Experiment and use it to predict
# Prediction is a sequence which is adjacent to the last date of the data, and does not exist in the data
# If you want to get more information about prediction, you can refer to code `exp/exp_informer.py function predict()` and `data/data_loader.py class Dataset_Pred`

exp = Exp(args)

#save_trained_model(exp, setting)

In [ ]:
########################################
# 2) Istanzia Exp_Informer
Exp = Exp_Informer
exp = Exp(args)

exp.predict(setting, True)

prediction = np.load('../results/'+setting+'/real_prediction.npy')

prediction.shape

model_file = os.path.join(args.checkpoints, setting, 'checkpoint.pth')
# o 'trained_model.pth' se quello è il checkpoint migliore

# 3) Carica i pesi
exp.model.load_state_dict(torch.load(model_file, map_location='cpu'))
if args.use_gpu:
    exp.model.cuda()
exp.model.eval()

# 4) Richiama test(...) 
#    Questo creerà i file pred.npy e true.npy in ./results/{setting}/
exp.test(setting)  

# 5) Carica i file generati
preds = np.load(f'../results/{setting}/pred.npy')
trues = np.load(f'../results/{setting}/true.npy')

print("Shape predizioni:", preds.shape)
print("Shape ground truth:", trues.shape)

# Nomi delle variabili (nel dataset)
variables = [f'Channel{i}' for i in range(1, 63)]

# Plot per ogni variabile
for i, var_name in enumerate(variables):
    idx_sample = 0  # Cambia questo per vedere altri esempi
    plt.figure(figsize=(8, 5))
    plt.plot(trues[idx_sample, :, i], label=f'GroundTruth ({var_name})')
    plt.plot(preds[idx_sample, :, i], label=f'Prediction ({var_name})')
    plt.title(f'Confronto - Variabile: {var_name} (Sample {idx_sample})')
    plt.xlabel('Time Step')
    plt.ylabel(var_name)
    plt.legend()
    plt.show()

Calcolo metriche

In [ ]:
# Funzioni metriche
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

for i, var_name in enumerate(variables):
    y_true = trues[:, :, i].flatten()  # Ground truth per la variabile i
    y_pred = preds[:, :, i].flatten()  # Predizioni per la variabile i
    
    print(f"\n--- Metriche per la variabile: {var_name} ---")
    print(f"MAE : {mae(y_true, y_pred):.4f}")
    print(f"MSE : {mse(y_true, y_pred):.4f}")
    print(f"RMSE: {rmse(y_true, y_pred):.4f}")